[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/lotto_645_sampling.ipynb)

# 로또 6/45, 모의추첨에서 1등을 한 번도 보지 못할 확률

45개 중 6개를 고른 모든 조합에 번호를 하나씩 붙입니다. 매 게임마다 균등하고 독립적으로 번호 하나를 뽑는 모형에서 한 게임의 1등 확률은 1 / C(45,6)입니다. [동행복권 게임 규칙](https://www.dhlottery.co.kr/lt645/intro)

## 한 번씩 뽑기와 묶음별로 뽑기

첫 실험은 조합 번호를 100만 번 뽑아 번호 0이 나온 횟수를 셉니다. 다음 실험은 이항분포로 한 묶음의 1등 횟수를 한꺼번에 뽑습니다. 묶음 크기마다 1만 번 반복한 뒤 한 번도 당첨되지 않은 묶음의 비율과 (1 − p)ⁿ을 비교합니다. 두 실험은 별도 난수 스트림을 사용합니다.

In [ ]:
import math
import os
import platform
from pathlib import Path
from urllib.request import urlopen

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np


SEED = 645
BATCHES = 10_000
COMBINATIONS = math.comb(45, 6)
P_WIN = 1 / COMBINATIONS
GAME_COUNTS = (100_000, 1_000_000, COMBINATIONS)


def simulate():
    # Independent streams keep the direct and batched experiments reproducible.
    direct_seed, batch_seed = np.random.SeedSequence(SEED).spawn(2)
    direct_rng = np.random.Generator(np.random.PCG64(direct_seed))
    batch_rng = np.random.Generator(np.random.PCG64(batch_seed))

    # Each unordered six-number set has one ID. By symmetry, fix the winner at 0.
    picks = direct_rng.integers(COMBINATIONS, size=1_000_000, dtype=np.int32)
    direct_hits = int(np.count_nonzero(picks == 0))

    rows = []
    for games in GAME_COUNTS:
        # One binomial sample is the win count across one batch of independent games.
        wins = batch_rng.binomial(games, P_WIN, size=BATCHES)
        zero_batches = int(np.count_nonzero(wins == 0))
        exact_zero = math.exp(games * math.log1p(-P_WIN))
        rows.append((games, zero_batches, exact_zero))
    return direct_hits, rows


def report(direct_hits, rows):
    assert COMBINATIONS == math.prod(range(40, 46)) // math.factorial(6) == 8_145_060
    print(f"Python {platform.python_version()} | NumPy {np.__version__} | Matplotlib {mpl.__version__}")
    print(f"seed={SEED} combinations={COMBINATIONS:,} batches={BATCHES:,}")
    print(f"Direct uniform combination IDs: {direct_hits} first-prize hits in 1,000,000 games")
    print("Games | Expected wins | Zero-hit batches | Observed zero (%) | Exact zero (%)")
    for games, zero_batches, exact_zero in rows:
        print(f"{games:,} | {games * P_WIN:.6f} | {zero_batches:,} | "
              f"{100 * zero_batches / BATCHES:.2f} | {100 * exact_zero:.4f}")

## 그래프

한국어 글꼴은 [Google Fonts NanumGothic](https://github.com/google/fonts/tree/main/ofl/nanumgothic)의 SIL Open Font License 1.1 배포본을 사용합니다. 그래프 언어는 기본값 ko이며, 실행 전에 환경 변수 CHART_LANG을 en 또는 both로 바꿀 수 있습니다.

In [ ]:
def register_korean_font():
    path = Path("cache/fonts/NanumGothic-Regular.ttf")
    if not path.exists():
        url = "https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf"
        with urlopen(url, timeout=30) as response:
            content = response.read()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
    font_manager.fontManager.addfont(path)
    return font_manager.FontProperties(fname=path).get_name()


def plot(rows, lang):
    labels = {
        "ko": ("첫 당첨 없이 끝난 묶음", "묶음당 독립 게임 수", "0회 당첨 비율 (%)",
               "이론값", "모의실험", ["10만", "100만", "8,145,060"]),
        "en": ("Batches with no first-prize wins", "Independent games per batch",
               "Batches with zero wins (%)", "Exact", "Simulation", ["100,000", "1,000,000", "8,145,060"]),
    }
    title, xlabel, ylabel, exact_label, sample_label, ticks = labels[lang]
    face = register_korean_font() if lang == "ko" else "DejaVu Sans"
    with plt.rc_context({"font.family": face, "figure.dpi": 145, "font.size": 12}):
        fig, ax = plt.subplots(figsize=(8, 4.8), layout="constrained")
        x = np.arange(len(rows))
        shares = [100 * zero / BATCHES for _, zero, _ in rows]
        bars = ax.bar(x, shares, width=0.48, color="#6eabb0", label=sample_label)
        expected = [100 * exact for _, _, exact in rows]
        for i, pct in enumerate(expected):
            ax.plot([i - 0.29, i + 0.29], [pct, pct], color="#28384c", lw=2.6,
                    label=exact_label if i == 0 else None)
        ax.bar_label(bars, labels=[f"{value:.2f}%" for value in shares], padding=5)
        ax.set(xticks=x, xticklabels=ticks, xlabel=xlabel, ylabel=ylabel, ylim=(0, 110))
        ax.set_yticks([0, 25, 50, 75, 100])
        ax.set_title(title, loc="left", pad=16)
        ax.grid(axis="y", alpha=0.18)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(loc="upper right", frameon=False)
        plt.show()


def main():
    chart_lang = os.environ.get("CHART_LANG", "ko")
    if chart_lang not in {"ko", "en", "both"}:
        raise ValueError("CHART_LANG must be ko, en, or both")
    direct_hits, rows = simulate()
    report(direct_hits, rows)
    for lang in (("ko", "en") if chart_lang == "both" else (chart_lang,)):
        plot(rows, lang)

In [ ]:
main()